In [1]:
# ============================================================
# PS S6E5 - exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min
#
# Base:
#   shared_007 TabM_D_Classifier
#
# Change from shared_007:
#   - Correct competition-only OOF
#   - Original rows are appended to each fold train side only
#   - Validation rows are competition train only
#   - Drop PitStop
#   - Drop IsOriginalData
#   - Drop Normalized_TyreLife from original
#   - Save OOF/pred artifacts for blend
#
# Purpose:
#   Test TabM as a new model family material.
#   Do not trust shared_007 reported CV because it used train+original combined CV.
# ============================================================

In [2]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 8.5 MB/s eta 0:00:00


In [3]:
import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

import torch

try:
    from pytabkit import TabM_D_Classifier
except Exception as e:
    raise RuntimeError(
        "pytabkit is not installed. In Kaggle, run: !pip install -q pytabkit"
    ) from e

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [4]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_PATH = Path(
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
    )

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5

    USE_ORIGINAL_ROWS = True

    # 021 safety policy
    DROP_PITSTOP = True
    DROP_IS_ORIGINAL_DATA = True

    # TabM params from shared_007
    TABM_PARAMS = {
        "arch_type": "tabm-normal",
        "tabm_k": 32,
        "num_emb_type": "pwl",
        "d_embedding": 32,
        "batch_size": 512,
        "lr": 7e-4,
        "n_epochs": 200,
        "dropout": 0.15,
        "d_block": 256,
        "n_blocks": 6,
        "weight_decay": 1e-2,
        "verbosity": 0,
    }

    PATIENCE = 5
    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


In [5]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int = CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def print_section(title: str):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            c_min, c_max = out[col].min(), out[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


def safe_divide(a, b, eps=1e-6):
    return a / (b + eps)


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


def safe_logloss(y_true, pred):
    return log_loss(y_true, clip_pred(pred))


seed_everything(CFG.SEED)

In [6]:
# ============================================================
# Load Data
# ============================================================

print_section("Load Data")

train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)
orig_df = pd.read_csv(CFG.ORIG_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)
print("orig_df :", orig_df.shape)

assert CFG.TARGET in train_df.columns
assert CFG.TARGET in orig_df.columns
assert CFG.ID_COL in train_df.columns
assert CFG.ID_COL in test_df.columns

if CFG.DANGER_COL in orig_df.columns:
    orig_df = orig_df.drop(columns=[CFG.DANGER_COL])
    print(f"Dropped danger column from original: {CFG.DANGER_COL}")

# Original may not have id. Add synthetic negative ids only for alignment.
if CFG.ID_COL not in orig_df.columns:
    orig_df[CFG.ID_COL] = -np.arange(1, len(orig_df) + 1)

train_df = reduce_mem_usage(train_df)
test_df = reduce_mem_usage(test_df)
orig_df = reduce_mem_usage(orig_df)

train_ids = train_df[CFG.ID_COL].copy()
test_ids = test_df[CFG.ID_COL].copy()

print("target mean competition:", train_df[CFG.TARGET].mean())
print("target mean original   :", orig_df[CFG.TARGET].mean())

assert CFG.DANGER_COL not in orig_df.columns


Load Data
train_df: (439140, 16)
test_df : (188165, 15)
orig_df : (101371, 16)
Dropped danger column from original: Normalized_TyreLife
target mean competition: 0.1989821
target mean original   : 0.25479673673930414


In [7]:
# ============================================================
# shared_007-style Feature Engineering
# ============================================================

BASE_CAT_COLS = ["Driver", "Compound", "Race"]

NUMERIC_BASE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]


def add_domain_features(df):
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedTotalLaps"] = safe_divide(
            out["LapNumber"],
            out["RaceProgress"].clip(lower=eps),
            eps,
        ).replace([np.inf, -np.inf], np.nan).clip(1, 120)

        out["LapsRemaining"] = (
            out["EstimatedTotalLaps"] - out["LapNumber"]
        ).clip(lower=0)

        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreAgeRatio"] = safe_divide(out["TyreLife"], out["LapNumber"].clip(lower=1), eps)
        out["LapPerTyreLife"] = safe_divide(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["LapMinusTyreLife"] = out["LapNumber"] - out["TyreLife"]
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if {"TyreLife", "EstimatedTotalLaps"}.issubset(out.columns):
        out["TyreAgeVsRace"] = safe_divide(out["TyreLife"], out["EstimatedTotalLaps"].clip(lower=1), eps)

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]
        out["TyreLife_x_RaceProgress"] = out["TyreLife"] * out["RaceProgress"]

    if {"TyreLife", "LapsRemaining"}.issubset(out.columns):
        out["TyreLife_to_LapsRemaining"] = safe_divide(out["TyreLife"], out["LapsRemaining"] + 1, eps)
        out["LapsRemaining_to_TyreLife"] = safe_divide(out["LapsRemaining"], out["TyreLife"] + 1, eps)

    if {"Cumulative_Degradation", "TyreLife"}.issubset(out.columns):
        out["DegPerTyreLap"] = safe_divide(out["Cumulative_Degradation"], out["TyreLife"].clip(lower=1), eps)
        out["AbsDegPerTyreLap"] = safe_divide(out["Cumulative_Degradation"].abs(), out["TyreLife"].clip(lower=1), eps)

    if {"Cumulative_Degradation", "LapNumber"}.issubset(out.columns):
        out["DegPerRaceLap"] = safe_divide(out["Cumulative_Degradation"], out["LapNumber"].clip(lower=1), eps)

    if {"LapTime_Delta", "TyreLife"}.issubset(out.columns):
        out["DeltaPerTyreLap"] = safe_divide(out["LapTime_Delta"], out["TyreLife"].clip(lower=1), eps)
        out["AbsDeltaPerTyreLap"] = safe_divide(out["LapTime_Delta"].abs(), out["TyreLife"].clip(lower=1), eps)

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()
        out["LapTimeDeltaPositive"] = (out["LapTime_Delta"] > 0).astype(np.int8)
        out["LapTimeDeltaNegative"] = (out["LapTime_Delta"] < 0).astype(np.int8)

    if "Cumulative_Degradation" in out.columns:
        out["Abs_Cumulative_Degradation"] = out["Cumulative_Degradation"].abs()
        out["Positive_Degradation"] = (out["Cumulative_Degradation"] > 0).astype(np.int8)

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if {"Position", "RaceProgress"}.issubset(out.columns):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]

    if {"Stint", "LapNumber"}.issubset(out.columns):
        out["Stint_x_LapNumber"] = out["Stint"] * out["LapNumber"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "RaceProgress" in out.columns:
        out["Early_Race"] = (out["RaceProgress"] <= 0.25).astype(np.int8)
        out["Mid_Race"] = (
            (out["RaceProgress"] > 0.25)
            & (out["RaceProgress"] <= 0.65)
        ).astype(np.int8)
        out["Late_Race"] = (out["RaceProgress"] > 0.65).astype(np.int8)

        out["RacePhase"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)

    if "LapNumber" in out.columns:
        out["LapBin"] = pd.cut(
            out["LapNumber"],
            bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
            labels=["L_000_005", "L_006_010", "L_011_020", "L_021_035", "L_036_050", "L_051_plus"],
        ).astype(str)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=["T_000_003", "T_004_007", "T_008_012", "T_013_020", "T_021_030", "T_031_plus"],
        ).astype(str)

    if "Position" in out.columns:
        out["PositionBin"] = pd.cut(
            out["Position"],
            bins=[-np.inf, 3, 8, 14, np.inf],
            labels=["front", "upper_mid", "lower_mid", "back"],
        ).astype(str)

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            val = out[cols[0]].astype(str)
            for c in cols[1:]:
                val = val + "_" + out[c].astype(str)
            out[name] = val

    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Compound_Stint", ["Compound", "Stint"])
    make_cross("Driver_Race", ["Driver", "Race"])
    make_cross("Driver_Compound", ["Driver", "Compound"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("Race_Compound_Stint", ["Race", "Compound", "Stint"])
    make_cross("Compound_RacePhase", ["Compound", "RacePhase"])
    make_cross("Compound_TyreLifeBin", ["Compound", "TyreLifeBin"])
    make_cross("RacePhase_TyreLifeBin", ["RacePhase", "TyreLifeBin"])

    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = out[col].astype(np.float32)

    return out

In [8]:
def add_digit_features(df, numeric_cols=None, int_digit_limit=3, decimal_digit_limit=2):
    out = df.copy()

    if numeric_cols is None:
        numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric_cols:
        if col not in out.columns:
            continue

        s = out[col].fillna(0).astype(float)
        abs_s = s.abs()

        for i in range(int_digit_limit):
            new_col = f"{col}_int_digit_{i + 1}"
            out[new_col] = ((abs_s // (10 ** i)) % 10).astype(np.int8)

        if pd.api.types.is_float_dtype(out[col]):
            for i in range(1, decimal_digit_limit + 1):
                new_col = f"{col}_dec_digit_{i}"
                out[new_col] = ((abs_s * (10 ** i)).round().astype(int) % 10).astype(np.int8)

    return out


def add_float_signature_features(df, float_cols=None):
    out = df.copy()

    if float_cols is None:
        float_cols = out.select_dtypes(include=["float32", "float64"]).columns.tolist()

    selected = [
        "RaceProgress",
        "LapTime (s)",
        "LapTime_Delta",
        "Cumulative_Degradation",
        "TyreAgeRatio",
        "DegPerTyreLap",
        "DegPerRaceLap",
        "DeltaPerTyreLap",
        "DeltaAbs",
        "PitWindowPressure",
        "EstimatedTotalLaps",
        "LapsRemaining",
        "LapMinusTyreLife",
    ]

    float_cols = [c for c in selected if c in out.columns]

    for col in float_cols:
        scaled = (out[col].fillna(0).astype(float) * 100).round().astype(int).abs()

        for i in range(5):
            new_col = f"{col}_sig_{i + 1}"
            digit = ((scaled // (10 ** i)) % 10).astype(np.int8)

            if digit.nunique() > 1:
                out[new_col] = digit.astype(str)

    return out


def add_string_precision_features(df):
    out = df.copy()

    if "RaceProgress" in out.columns:
        out["RaceProgress_str"] = out["RaceProgress"].round(4).astype(str)

    if "EstimatedTotalLaps" in out.columns:
        out["EstimatedTotalLaps_str"] = out["EstimatedTotalLaps"].round(1).astype(str)

    if "TyreAgeRatio" in out.columns:
        out["TyreAgeRatio_str"] = out["TyreAgeRatio"].round(3).astype(str)

    return out

In [9]:
def add_frequency_features(train_df, test_df, original_df=None):
    freq_cols = [
        "Driver",
        "Race",
        "Compound",
        "Race_Year",
        "Compound_Stint",
        "Driver_Race",
        "Driver_Compound",
        "Race_Compound",
        "Race_Compound_Stint",
        "Compound_RacePhase",
        "Compound_TyreLifeBin",
        "RacePhase_TyreLifeBin",
        "LapBin",
        "TyreLifeBin",
        "PositionBin",
    ]

    freq_cols = [c for c in freq_cols if c in train_df.columns and c in test_df.columns]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    base = pd.concat([f[freq_cols] for f in frames], axis=0, ignore_index=True)
    total = len(base)

    for col in freq_cols:
        counts = base[col].astype(str).value_counts(dropna=False)

        for f in frames:
            f[f"{col}_count"] = f[col].astype(str).map(counts).fillna(0).astype(np.float32)
            f[f"{col}_freq"] = (f[f"{col}_count"] / total).astype(np.float32)

    del base
    gc.collect()

    if original_df is not None:
        return train_df, test_df, original_df

    return train_df, test_df, None


def add_light_group_stats(train_df, test_df, original_df=None):
    group_cols_list = [
        ["Race_Year"],
        ["Race_Compound_Stint"],
        ["Driver_Race"],
        ["Compound_Stint"],
    ]

    value_cols = [
        "LapTime_Delta",
        "Position_Change",
        "RaceProgress",
        "TyreLife",
    ]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    for group_cols in group_cols_list:
        group_cols = [c for c in group_cols if all(c in f.columns for f in frames)]

        if not group_cols:
            continue

        group_name = "_".join(group_cols)

        for value_col in value_cols:
            if not all(value_col in f.columns for f in frames):
                continue

            base = pd.concat(
                [f[group_cols + [value_col]] for f in frames],
                axis=0,
                ignore_index=True,
            )

            stats = (
                base.groupby(group_cols, dropna=False)[value_col]
                .agg(["mean", "std"])
                .reset_index()
            )

            stats.columns = group_cols + [
                f"{value_col}_mean_by_{group_name}",
                f"{value_col}_std_by_{group_name}",
            ]

            for idx, f in enumerate(frames):
                frames[idx] = f.merge(stats, on=group_cols, how="left")

                mean_col = f"{value_col}_mean_by_{group_name}"
                frames[idx][f"{value_col}_diff_mean_by_{group_name}"] = (
                    frames[idx][value_col] - frames[idx][mean_col]
                ).astype(np.float32)

            del base, stats
            gc.collect()

    if original_df is not None:
        return frames[0], frames[1], frames[2]

    return frames[0], frames[1], None

In [10]:
def clean_columns(train_df, test_df, original_df=None):
    train_df = train_df.copy()
    test_df = test_df.copy()

    if original_df is not None:
        original_df = original_df.copy()

    common_cols = [c for c in train_df.columns if c in test_df.columns]

    train_df = train_df[common_cols + [CFG.TARGET]]
    test_df = test_df[common_cols]

    if original_df is not None:
        for c in common_cols:
            if c not in original_df.columns:
                original_df[c] = np.nan

        original_df = original_df[common_cols + [CFG.TARGET]]

    return train_df, test_df, original_df


def fill_missing_and_types(train_df, test_df, original_df=None):
    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    all_feature_cols = [c for c in train_df.columns if c != CFG.TARGET]

    cat_cols = []

    for col in all_feature_cols:
        if any(
            (
                f[col].dtype == "object"
                or str(f[col].dtype).startswith("category")
                or str(f[col].dtype).startswith("string")
            )
            for f in frames
            if col in f.columns
        ):
            cat_cols.append(col)

    num_cols = [c for c in all_feature_cols if c not in cat_cols]

    for col in cat_cols:
        values = pd.concat([f[col].astype("string") for f in frames if col in f.columns], axis=0)
        mode_value = values.mode().iloc[0] if len(values.mode()) else "__MISSING__"

        for f in frames:
            if col in f.columns:
                f[col] = f[col].astype("string").fillna(mode_value).astype(str)

    for col in num_cols:
        values = pd.concat([f[col] for f in frames if col in f.columns], axis=0)
        fill_value = values.median()

        for f in frames:
            if col in f.columns:
                f[col] = f[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)
                if f[col].dtype == "float64":
                    f[col] = f[col].astype(np.float32)

    train_df = reduce_mem_usage(train_df)
    test_df = reduce_mem_usage(test_df)

    if original_df is not None:
        original_df = reduce_mem_usage(original_df)

    return train_df, test_df, original_df, cat_cols

In [11]:
# ============================================================
# Feature Engineering
# ============================================================

print_section("Feature Engineering")

train_fe = train_df.copy()
test_fe = test_df.copy()
orig_fe = orig_df.copy()

# 021: Do not use IsOriginalData
# shared_007 originally created IsOriginalData, but 021 explicitly drops it.
# We simply do not create it.

# Add domain features
train_fe = add_domain_features(train_fe)
test_fe = add_domain_features(test_fe)
orig_fe = add_domain_features(orig_fe)

# Add digit features
digit_source_cols = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
    "EstimatedTotalLaps",
    "LapsRemaining",
    "TyreAgeRatio",
    "DegPerTyreLap",
    "DegPerRaceLap",
    "DeltaPerTyreLap",
    "DeltaAbs",
    "PositionPressure",
    "StintPressure",
    "PitWindowPressure",
    "LapMinusTyreLife",
]

digit_source_cols = [
    c for c in digit_source_cols
    if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns
]

train_fe = add_digit_features(train_fe, digit_source_cols)
test_fe = add_digit_features(test_fe, digit_source_cols)
orig_fe = add_digit_features(orig_fe, digit_source_cols)

# shared_007 defined this but did not call it in the pasted code.
# We keep it off for closer behavior unless you intentionally want extra features.
# train_fe = add_float_signature_features(train_fe)
# test_fe = add_float_signature_features(test_fe)
# orig_fe = add_float_signature_features(orig_fe)

train_fe = add_string_precision_features(train_fe)
test_fe = add_string_precision_features(test_fe)
orig_fe = add_string_precision_features(orig_fe)

train_fe, test_fe, orig_fe = add_frequency_features(train_fe, test_fe, orig_fe)
train_fe, test_fe, orig_fe = add_light_group_stats(train_fe, test_fe, orig_fe)

train_fe, test_fe, orig_fe = clean_columns(train_fe, test_fe, orig_fe)
train_fe, test_fe, orig_fe, cat_cols = fill_missing_and_types(train_fe, test_fe, orig_fe)

print("Train FE shape   :", train_fe.shape)
print("Test FE shape    :", test_fe.shape)
print("Original FE shape:", orig_fe.shape)
print("Categorical features:", len(cat_cols))

assert CFG.DANGER_COL not in train_fe.columns
assert CFG.DANGER_COL not in test_fe.columns
assert CFG.DANGER_COL not in orig_fe.columns
assert "IsOriginalData" not in train_fe.columns
assert "IsOriginalData" not in test_fe.columns
assert "IsOriginalData" not in orig_fe.columns


Feature Engineering
Train FE shape   : (439140, 244)
Test FE shape    : (188165, 243)
Original FE shape: (101371, 244)
Categorical features: 19


In [12]:
# ============================================================
# Feature Selection and Encoding
# ============================================================

print_section("Feature Selection and Encoding")

DROP_FROM_X = [
    CFG.ID_COL,
    CFG.TARGET,
]

if CFG.DROP_PITSTOP:
    DROP_FROM_X.append("PitStop")

if CFG.DROP_IS_ORIGINAL_DATA:
    DROP_FROM_X.append("IsOriginalData")

FEATURES = [
    c for c in train_fe.columns
    if c not in DROP_FROM_X
    and c in test_fe.columns
    and c in orig_fe.columns
]

cat_cols = [c for c in cat_cols if c in FEATURES]

print("feature count before encoding:", len(FEATURES))
print("cat_cols:", len(cat_cols))
print("DROP_FROM_X:", DROP_FROM_X)

assert "PitStop" not in FEATURES
assert "IsOriginalData" not in FEATURES
assert CFG.TARGET not in FEATURES
assert CFG.ID_COL not in FEATURES
assert CFG.DANGER_COL not in FEATURES

# Fit encoder using train + orig + test category vocabulary.
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

encode_base = pd.concat(
    [
        train_fe[cat_cols],
        orig_fe[cat_cols],
        test_fe[cat_cols],
    ],
    axis=0,
    ignore_index=True,
).astype(str)

encoder.fit(encode_base)

for df in [train_fe, orig_fe, test_fe]:
    df[cat_cols] = encoder.transform(df[cat_cols].astype(str))

# Ensure numeric
for df in [train_fe, orig_fe, test_fe]:
    for col in FEATURES:
        if df[col].dtype == "object" or str(df[col].dtype).startswith("category") or str(df[col].dtype).startswith("string"):
            codes, _ = pd.factorize(df[col].astype(str), sort=False)
            df[col] = codes.astype(np.int32)
        elif df[col].dtype == "float64":
            df[col] = df[col].astype(np.float32)

feature_df = pd.DataFrame({
    "feature": FEATURES,
    "is_cat": [c in cat_cols for c in FEATURES],
    "is_digit": [
        ("_int_digit_" in c) or ("_dec_digit_" in c)
        for c in FEATURES
    ],
    "is_freq": [
        c.endswith("_count") or c.endswith("_freq")
        for c in FEATURES
    ],
    "is_group_stat": [
        ("_mean_by_" in c) or ("_std_by_" in c) or ("_diff_mean_by_" in c)
        for c in FEATURES
    ],
    "contains_year": ["Year" in c for c in FEATURES],
    "contains_pitstop": ["PitStop" in c for c in FEATURES],
    "contains_original": ["IsOriginalData" in c for c in FEATURES],
})

display(feature_df.head())
print("Any PitStop feature:", feature_df["contains_pitstop"].any())
print("Any IsOriginalData feature:", feature_df["contains_original"].any())

# Note:
# contains_pitstop may be True if derived feature names include PitStop-like strings.
# In this implementation the raw PitStop column is removed.


Feature Selection and Encoding
feature count before encoding: 241
cat_cols: 19
DROP_FROM_X: ['id', 'PitNextLap', 'PitStop', 'IsOriginalData']


,feature,is_cat,is_digit,is_freq,is_group_stat,contains_year,contains_pitstop,contains_original
0,Driver,True,False,False,False,False,False,False
1,Compound,True,False,False,False,False,False,False
2,Race,True,False,False,False,False,False,False
3,Year,False,False,False,False,True,False,False
4,LapNumber,False,False,False,False,False,False,False


Any PitStop feature: True
Any IsOriginalData feature: False


In [13]:
# ============================================================
# CV Setup
# ============================================================

print_section("CV Setup")

X_comp = train_fe[FEATURES].copy()
y_comp = train_fe[CFG.TARGET].astype(int).reset_index(drop=True)

X_orig = orig_fe[FEATURES].copy()
y_orig = orig_fe[CFG.TARGET].astype(int).reset_index(drop=True)

X_test = test_fe[FEATURES].copy()

# competition-only folds
strat_key = train_fe[CFG.TARGET].astype(str) + "__" + train_fe["Year"].astype(str)

folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_comp, strat_key)
)

for i, (_, va_idx) in enumerate(folds, 1):
    print(
        f"fold {i}: valid rows={len(va_idx)}, valid rate={y_comp.iloc[va_idx].mean():.6f}"
    )

print("X_comp:", X_comp.shape)
print("X_orig:", X_orig.shape)
print("X_test:", X_test.shape)


CV Setup
fold 1: valid rows=87828, valid rate=0.198991
fold 2: valid rows=87828, valid rate=0.198991
fold 3: valid rows=87828, valid rate=0.198968
fold 4: valid rows=87828, valid rate=0.198968
fold 5: valid rows=87828, valid rate=0.198991
X_comp: (439140, 241)
X_orig: (101371, 241)
X_test: (188165, 241)


In [14]:
# ============================================================
# TabM Training - competition-only OOF
# ============================================================

print_section("TabM Training")

start_time = time.time()

oof = np.zeros(len(X_comp), dtype=np.float32)
pred = np.zeros(len(X_test), dtype=np.float32)

fold_records = []
best_models = []
scalers = []

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    print("\n" + "=" * 90)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 90)

    X_tr_comp = X_comp.iloc[tr_idx].copy().reset_index(drop=True)
    y_tr_comp = y_comp.iloc[tr_idx].copy().reset_index(drop=True)

    X_va = X_comp.iloc[va_idx].copy().reset_index(drop=True)
    y_va = y_comp.iloc[va_idx].copy().reset_index(drop=True)

    if CFG.USE_ORIGINAL_ROWS:
        X_tr = pd.concat(
            [X_tr_comp, X_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_comp, y_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_comp
        y_tr = y_tr_comp

    X_te = X_test.copy().reset_index(drop=True)

    print("X_tr:", X_tr.shape, "y_tr mean:", float(y_tr.mean()))
    print("X_va:", X_va.shape, "y_va mean:", float(y_va.mean()))
    print("X_te:", X_te.shape)

    # Scaling
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_tr).astype(np.float32)
    X_valid_scaled = scaler.transform(X_va).astype(np.float32)
    X_test_scaled = scaler.transform(X_te).astype(np.float32)

    X_train_scaled = np.clip(X_train_scaled, -5, 5)
    X_valid_scaled = np.clip(X_valid_scaled, -5, 5)
    X_test_scaled = np.clip(X_test_scaled, -5, 5)

    seed_everything(CFG.SEED + fold)

    model = TabM_D_Classifier(
        random_state=CFG.SEED,
        device=DEVICE,
        patience=CFG.PATIENCE,
        **CFG.TABM_PARAMS,
    )

    model.fit(X_train_scaled, y_tr)

    va_pred = clip_pred(model.predict_proba(X_valid_scaled)[:, 1]).astype(np.float32)
    te_pred = clip_pred(model.predict_proba(X_test_scaled)[:, 1]).astype(np.float32)

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_logloss = safe_logloss(y_va, va_pred)

    print(f"Fold {fold} AUC    : {fold_auc:.9f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.9f}")

    fold_records.append({
        "fold": fold,
        "auc": float(fold_auc),
        "logloss": float(fold_logloss),
        "n_train_comp": int(len(X_tr_comp)),
        "n_train_orig": int(len(X_orig)) if CFG.USE_ORIGINAL_ROWS else 0,
        "n_valid": int(len(X_va)),
        "n_features": int(len(FEATURES)),
        "target_mean_train": float(y_tr.mean()),
        "target_mean_valid": float(y_va.mean()),
    })

    best_models.append(model)
    scalers.append(scaler)

    del X_tr_comp, y_tr_comp, X_va, y_va, X_tr, y_tr, X_te
    del X_train_scaled, X_valid_scaled, X_test_scaled, va_pred, te_pred
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

elapsed_sec = time.time() - start_time


TabM Training

Fold 1/5
X_tr: (452683, 241) y_tr mean: 0.21147911452385001
X_va: (87828, 241) y_va mean: 0.19899121009245344
X_te: (188165, 241)
Fold 1 AUC    : 0.952180881
Fold 1 LogLoss: 0.220654972

Fold 2/5
X_tr: (452683, 241) y_tr mean: 0.21147911452385001
X_va: (87828, 241) y_va mean: 0.19899121009245344
X_te: (188165, 241)
Fold 2 AUC    : 0.951356151
Fold 2 LogLoss: 0.221460160

Fold 3/5
X_tr: (452683, 241) y_tr mean: 0.21148353262658418
X_va: (87828, 241) y_va mean: 0.1989684383112447
X_te: (188165, 241)
Fold 3 AUC    : 0.951502723
Fold 3 LogLoss: 0.221221190

Fold 4/5
X_tr: (452683, 241) y_tr mean: 0.21148353262658418
X_va: (87828, 241) y_va mean: 0.1989684383112447
X_te: (188165, 241)
Fold 4 AUC    : 0.950752703
Fold 4 LogLoss: 0.222409766

Fold 5/5
X_tr: (452683, 241) y_tr mean: 0.21147911452385001
X_va: (87828, 241) y_va mean: 0.19899121009245344
X_te: (188165, 241)
Fold 5 AUC    : 0.952179022
Fold 5 LogLoss: 0.219998351


In [15]:
# ============================================================
# CV Result
# ============================================================

print_section("CV Result")

cv_auc = roc_auc_score(y_comp, oof)
cv_logloss = safe_logloss(y_comp, oof)

fold_df = pd.DataFrame(fold_records)

print(f"CV AUC    : {cv_auc:.9f}")
print(f"CV LogLoss: {cv_logloss:.9f}")
print("fold mean:", fold_df["auc"].mean())
print("fold std :", fold_df["auc"].std())
print("elapsed sec:", elapsed_sec)

display(fold_df)


CV Result
CV AUC    : 0.951491547
CV LogLoss: 0.221148888
fold mean: 0.9515942958676625
fold std : 0.0006040208046452686
elapsed sec: 19632.509905338287


,fold,auc,logloss,n_train_comp,n_train_orig,n_valid,n_features,target_mean_train,target_mean_valid
0,1,0.952181,0.220655,351312,101371,87828,241,0.211479,0.198991
1,2,0.951356,0.221460,351312,101371,87828,241,0.211479,0.198991
2,3,0.951503,0.221221,351312,101371,87828,241,0.211484,0.198968
3,4,0.950753,0.222410,351312,101371,87828,241,0.211484,0.198968
4,5,0.952179,0.219998,351312,101371,87828,241,0.211479,0.198991


In [16]:
# ============================================================
# Save Artifacts
# ============================================================

print_section("Save Artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

oof_csv = pd.DataFrame({
    CFG.ID_COL: train_ids.values,
    CFG.TARGET: y_comp.values,
    "pred": oof,
})
oof_csv.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub_out = sub.copy()
target_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[target_col] = clip_pred(pred)

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

fold_df.to_csv(CFG.OUTDIR / "fold_scores.csv", index=False)
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

pd.Series(FEATURES, name="feature").to_csv(CFG.OUTDIR / "features.txt", index=False)
pd.Series(cat_cols, name="cat_feature").to_csv(CFG.OUTDIR / "categorical_features.csv", index=False)

print("Saved to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(oof_csv.head())
display(sub_out.head())
display(feature_df.head(50))


Save Artifacts
Saved to: /kaggle/working/exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min
Submission: /kaggle/working/exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min/submission_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.csv


,id,PitNextLap,pred
0,0,1,0.772774
1,1,0,0.630055
2,2,1,0.698714
3,3,0,0.003275
4,4,0,0.313379


,id,PitNextLap
0,439140,0.008356
1,439141,0.013984
2,439142,0.003395
3,439143,0.325098
4,439144,0.846494


,feature,is_cat,is_digit,is_freq,is_group_stat,contains_year,contains_pitstop,contains_original
0,Driver,True,False,False,False,False,False,False
1,Compound,True,False,False,False,False,False,False
2,Race,True,False,False,False,False,False,False
3,Year,False,False,False,False,True,False,False
4,LapNumber,False,False,False,False,False,False,False
5,Stint,False,False,False,False,False,False,False
6,TyreLife,False,False,False,False,False,False,False
7,Position,False,False,False,False,False,False,False
8,LapTime (s),False,False,False,False,False,False,False
9,LapTime_Delta,False,False,False,False,False,False,False


In [17]:
# ============================================================
# Save cv_result.json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "source": {
        "shared_code": "shared_007",
        "model_component": "TabM_D_Classifier",
        "conversion": "competition-only OOF with original rows appended only to fold train",
        "note": [
            "The original shared_007 CV is not comparable because it performs CV on train+original combined rows.",
            "021 fixes the evaluation by using competition train folds only for OOF.",
            "Original rows are appended to fold train side only.",
            "Validation rows are always competition train only.",
            "PitStop is dropped.",
            "IsOriginalData is not created and not used.",
            "Normalized_TyreLife is dropped from original.",
        ],
    },
    "data": {
        "train_shape": list(train_df.shape),
        "test_shape": list(test_df.shape),
        "original_shape_after_drop": list(orig_df.shape),
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL_ROWS,
        "competition_target_mean": float(train_df[CFG.TARGET].mean()),
        "original_target_mean": float(orig_df[CFG.TARGET].mean()),
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "stratification": "target x Year",
        "n_splits": CFG.N_SPLITS,
        "shuffle": True,
        "random_state": CFG.SEED,
        "validation_scope": "competition train only",
    },
    "features": {
        "feature_count": int(len(FEATURES)),
        "categorical_feature_count": int(len(cat_cols)),
        "drop_from_x": DROP_FROM_X,
        "pitstop_used": "PitStop" in FEATURES,
        "isoriginaldata_used": "IsOriginalData" in FEATURES,
        "normalized_tyrelife_policy": [
            "Normalized_TyreLife is dropped.",
            "Direct use is forbidden.",
            "Intentional proxy reconstruction is not added.",
        ],
        "feature_blocks": [
            "shared007 domain features",
            "digit features",
            "string precision features",
            "frequency features",
            "light group stats",
            "ordinal encoded categorical features",
            "standard scaling before TabM",
        ],
        "risk_features_to_watch": [
            "Race_Year",
            "TyreAgeRatio_str",
            "LapTime_Delta group stats",
            "Position_Change group stats",
            "Race_Year group stats",
        ],
    },
    "model": {
        "family": "TabM",
        "estimator": "pytabkit.TabM_D_Classifier",
        "device": DEVICE,
        "params": CFG.TABM_PARAMS,
        "patience": CFG.PATIENCE,
    },
    "result": {
        "cv_auc": float(cv_auc),
        "cv_logloss": float(cv_logloss),
        "fold_auc_mean": float(fold_df["auc"].mean()),
        "fold_auc_std": float(fold_df["auc"].std()),
        "fold_scores": fold_records,
        "elapsed_sec": float(elapsed_sec),
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "fold_scores": str(CFG.OUTDIR / "fold_scores.csv"),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "categorical_features": str(CFG.OUTDIR / "categorical_features.csv"),
        "features": str(CFG.OUTDIR / "features.txt"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
    "decision_policy": {
        "single_model": [
            "Do not compare against shared_007 reported CV 0.96343.",
            "Compare against existing valid OOF models only.",
            "If valid competition-only CV is around 0.951-0.953, it may still be useful as TabM diversity.",
        ],
        "blend": [
            "Add 021 to the current best stack: 007/009/014/015/016/017/018/020.",
            "Evaluate avg / Ridge / ElasticNet / LogReg / HC / NM.",
            "If 021 receives positive weight and improves CV/LB, keep TabM branch.",
            "If 021 is zero-weight, close shared_007 branch.",
        ],
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "experiment": {
    "id": "exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min",
    "competition": "PS S6E5 Predicting F1 Pit Stops",
    "metric": "AUC"
  },
  "source": {
    "shared_code": "shared_007",
    "model_component": "TabM_D_Classifier",
    "conversion": "competition-only OOF with original rows appended only to fold train",
    "note": [
      "The original shared_007 CV is not comparable because it performs CV on train+original combined rows.",
      "021 fixes the evaluation by using competition train folds only for OOF.",
      "Original rows are appended to fold train side only.",
      "Validation rows are always competition train only.",
      "PitStop is dropped.",
      "IsOriginalData is not created and not used.",
      "Normalized_TyreLife is dropped from original."
    ]
  },
  "data": {
    "train_shape": [
      439140,
      16
    ],
    "test_shape": [
      188165,
      15
    ],
    "original_shape_after_drop": [
      101371,

In [18]:
# ============================================================
# Final Preview
# ============================================================

print_section("Final Preview")

print("CV AUC:", cv_auc)
print("CV LogLoss:", cv_logloss)
print("OOF range:", float(oof.min()), float(oof.max()), "mean:", float(oof.mean()))
print("Pred range:", float(pred.min()), float(pred.max()), "mean:", float(pred.mean()))

display(fold_df)
display(oof_csv.head())
display(sub_out.head())


Final Preview
CV AUC: 0.9514915466418662
CV LogLoss: 0.22114888779775163
OOF range: 9.999999974752427e-07 0.9994305968284607 mean: 0.20802004635334015
Pred range: 3.1396998565469403e-06 0.9967510104179382 mean: 0.20708689093589783


,fold,auc,logloss,n_train_comp,n_train_orig,n_valid,n_features,target_mean_train,target_mean_valid
0,1,0.952181,0.220655,351312,101371,87828,241,0.211479,0.198991
1,2,0.951356,0.221460,351312,101371,87828,241,0.211479,0.198991
2,3,0.951503,0.221221,351312,101371,87828,241,0.211484,0.198968
3,4,0.950753,0.222410,351312,101371,87828,241,0.211484,0.198968
4,5,0.952179,0.219998,351312,101371,87828,241,0.211479,0.198991


,id,PitNextLap,pred
0,0,1,0.772774
1,1,0,0.630055
2,2,1,0.698714
3,3,0,0.003275
4,4,0,0.313379


,id,PitNextLap
0,439140,0.008356
1,439141,0.013984
2,439142,0.003395
3,439143,0.325098
4,439144,0.846494
